# Session 4: Coordinated Multiple Views

Vitessce is built around a concept called **coordinated multiple views**: every view in the dashboard shares a common **coordination space** — a dictionary of named state variables called *coordination types*. When a user interacts with one view (selects a gene, highlights a cell cluster, zooms in), the coordination space updates and all subscribed views respond in concert.

In this notebook you will learn to:

1. Understand the coordination space and how views share state by default
2. Use `config.link_views()` to set initial coordination values and explicitly link views
3. Use `config.add_coordination()` to create *separate* coordination scopes so two views can track *independent* state
4. Use `config.link_views_by_dict()` for advanced nested coordination
5. Build a linked spatial + single-cell dashboard

In [ ]:
!pip install "vitessce[all]==3.9.2" "scanpy" "easy-vitessce==0.0.11" "spatialdata==0.7.3" "spatialdata-plot==0.4.0"

In [ ]:
## 1. How the coordination space works

Every view in Vitessce is connected to the coordination space through **coordination scopes** — named slots that hold a single coordination value. Multiple views can share the same scope, which is what makes them "coordinated".

```
Coordination Space
───────────────────────────────────────────
featureSelection["A"]  = ["CD79A"]    ← Views 1 and 2 both read scope "A"
obsSetSelection["A"]   = [["Monocyte"]]
spatialZoom["A"]       = -2
...
```

When a user clicks a gene in the Feature List view, it writes to `featureSelection["A"]`. Both the UMAP scatterplot and the Heatmap are subscribed to `featureSelection["A"]`, so they both immediately re-color.

### Default behavior

By default, **all views sharing the same dataset are wired to the same coordination scopes** (scope `"A"` for most types). This is why clicking a cell type in the OBS_SETS panel automatically highlights cells in every other view — you get coordination for free.

## 2. Setting initial coordination values with `link_views()`

The `link_views()` method sets the **initial value** of one or more coordination types for a group of views. All listed views will share the same coordination scope for the listed types.

```python
config.link_views(
    views=[view1, view2],
    coordination_types=["featureSelection", "obsColorEncoding"],
    coordination_values=[["CD79A"], "geneSelection"]
)
```

This is the most common way to:
- Pre-select a gene so the visualization opens with expression coloring already visible
- Pre-zoom the spatial view to a region of interest
- Set the initial cell type selection

In [ ]:
import os
from os.path import join, isdir
import scanpy as sc
from vitessce import VitessceConfig, ViewType as vt, AnnDataWrapper
from vitessce.data_utils import optimize_adata, VAR_CHUNK_SIZE

# Prepare data
adata = sc.datasets.pbmc68k_reduced()
zarr_path = join("data", "pbmc68k.zarr")
if not isdir(zarr_path):
    os.makedirs("data", exist_ok=True)
    adata_opt = optimize_adata(
        adata,
        obs_cols=["bulk_labels", "louvain"],
        obsm_keys=["X_umap", "X_pca"],
        optimize_X=True,
    )
    adata_opt.write_zarr(zarr_path, chunks=[adata_opt.shape[0], VAR_CHUNK_SIZE])

# Build config
vc = VitessceConfig(schema_version="1.0.15", name="Linked Views — Initial Gene Selection")
dataset = vc.add_dataset(name="PBMC").add_object(AnnDataWrapper(
    adata_store=zarr_path,
    obs_set_paths=["obs/bulk_labels"],
    obs_set_names=["Cell Type"],
    obs_embedding_paths=["obsm/X_umap"],
    obs_embedding_names=["UMAP"],
    obs_feature_matrix_path="X",
))

umap     = vc.add_view(vt.SCATTERPLOT, dataset=dataset, mapping="UMAP")
obs_sets = vc.add_view(vt.OBS_SETS, dataset=dataset)
genes    = vc.add_view(vt.FEATURE_LIST, dataset=dataset)
heatmap  = vc.add_view(vt.HEATMAP, dataset=dataset)

# Pre-select CD79A and use gene-expression coloring from the start
vc.link_views(
    views=[umap, heatmap],
    coordination_types=["featureSelection", "obsColorEncoding"],
    coordination_values=[["CD79A"], "geneSelection"],
)

vc.layout((umap | (obs_sets / genes)) / heatmap)
vc.widget()

## 3. Independent coordination scopes with `add_coordination()`

Sometimes you want two views to be **independent** — for example, two scatterplots each showing a different gene simultaneously, so you can compare expression patterns side-by-side.

By default, two scatterplots share the same `featureSelection` scope. To give each its own independent scope, use `add_coordination()` to create separate scopes and `view.use_coordination()` to bind each view to its own scope:

```python
[scope_b] = config.add_coordination("featureSelection")
scope_b.set_value(["LYZ"])            # initial value for this scope
umap_b.use_coordination(scope_b)      # bind this view to scope_b only
```

The example below creates two UMAPs: the left one starts on `CD79A` (a B-cell marker), the right one starts on `LYZ` (a monocyte marker). They can be navigated independently.

In [ ]:
vc2 = VitessceConfig(schema_version="1.0.15", name="Independent Gene Comparison")
dataset2 = vc2.add_dataset(name="PBMC").add_object(AnnDataWrapper(
    adata_store=zarr_path,
    obs_set_paths=["obs/bulk_labels"],
    obs_set_names=["Cell Type"],
    obs_embedding_paths=["obsm/X_umap"],
    obs_embedding_names=["UMAP"],
    obs_feature_matrix_path="X",
))

umap_a = vc2.add_view(vt.SCATTERPLOT, dataset=dataset2, mapping="UMAP")
umap_b = vc2.add_view(vt.SCATTERPLOT, dataset=dataset2, mapping="UMAP")

# Give each scatterplot its own independent featureSelection scope
[scope_a] = vc2.add_coordination("featureSelection")
scope_a.set_value(["CD79A"])    # B-cell marker
[scope_b] = vc2.add_coordination("featureSelection")
scope_b.set_value(["LYZ"])      # Monocyte marker

umap_a.use_coordination(scope_a)
umap_b.use_coordination(scope_b)

# Both plots color by gene expression
vc2.link_views([umap_a, umap_b], ["obsColorEncoding"], ["geneSelection"])

vc2.layout(umap_a | umap_b)
vc2.widget()

## 4. Advanced coordination with `link_views_by_dict()`

`link_views_by_dict()` accepts a dictionary of coordination values and supports **nested coordination** via `CoordinationLevel` (CL). This is the mechanism used for image-layer properties like channel colors, which are themselves nested structures.

The pattern:
```python
from vitessce import CoordinationLevel as CL, get_initial_coordination_scope_prefix

vc.link_views_by_dict(
    [spatial, layer_controller],
    {
        "imageLayer": CL([{
            "photometricInterpretation": "BlackIsZero",
            "imageChannel": CL([
                {"spatialTargetC": 0, "spatialChannelColor": [255, 0, 0]},
                {"spatialTargetC": 1, "spatialChannelColor": [0, 255, 0]},
            ]),
        }]),
    },
    scope_prefix=get_initial_coordination_scope_prefix("A", "image"),
)
```

You already saw this in Session 1 for the multiplexed image. For most beginner use cases `link_views()` is sufficient; `link_views_by_dict()` becomes useful when configuring image channels or other deeply nested state.

## 5. Linked spatial + single-cell dashboard

The most powerful use of coordination is linking a **spatial view** and a **scatterplot** so that:

- Selecting a cell type in the OBS_SETS panel highlights those cells in both the tissue image and the UMAP
- Selecting a gene in the FEATURE_LIST colors both the Visium spots and the UMAP points by that gene's expression

We achieve this by building a `SpatialDataWrapper` that exposes both the spatial coordinates and the gene expression matrix, and adding both spatial and non-spatial views to the same configuration.

The Visium dataset used here (breast cancer, 10x Genomics) has ~3,800 spots, each with expression measurements for ~16,000 genes.

In [ ]:
import zipfile
from os.path import isfile, isdir
from urllib.request import urlretrieve

data_dir = "data"
sdata_filepath = join(data_dir, "visium.spatialdata.zarr")

if not isdir(sdata_filepath):
    zip_filepath = join(data_dir, "visium.spatialdata.zarr.zip")
    if not isfile(zip_filepath):
        os.makedirs(data_dir, exist_ok=True)
        urlretrieve(
            "https://data-2.vitessce.io/sdata-datasets/visium.spatialdata.zarr.zip",
            zip_filepath,
        )
    with zipfile.ZipFile(zip_filepath, "r") as z:
        z.extractall(data_dir)
        os.rename(join(data_dir, "data.zarr"), sdata_filepath)

print("Dataset ready.")

In [ ]:
from vitessce import SpatialDataWrapper

vc3 = VitessceConfig(schema_version="1.0.18", name="Linked Spatial + Single-Cell Dashboard")

wrapper = SpatialDataWrapper(
    sdata_path=sdata_filepath,
    image_path="images/ST8059050_hires_image",
    obs_spots_path="shapes/ST8059050",
    table_path="tables/table",
    obs_feature_matrix_path="tables/table/X",
    region="ST8059050",
    coordinate_system="global",
    coordination_values={"obsType": "spot"},
)
dataset3 = vc3.add_dataset(name="Visium Breast Cancer").add_object(wrapper)

# --- Views ---
spatial3   = vc3.add_view("spatialBeta", dataset=dataset3)
lc3        = vc3.add_view("layerControllerBeta", dataset=dataset3)
obs_sets3  = vc3.add_view(vt.OBS_SETS, dataset=dataset3)
genes3     = vc3.add_view(vt.FEATURE_LIST, dataset=dataset3)
heatmap3   = vc3.add_view(vt.HEATMAP, dataset=dataset3)

# Share obsType across all views
vc3.link_views(
    [spatial3, lc3, obs_sets3, genes3, heatmap3],
    ["obsType"],
    [wrapper.obs_type_label],
)

# Pre-select a gene so the spatial spots open with expression coloring visible
vc3.link_views(
    [spatial3, heatmap3],
    ["featureSelection", "obsColorEncoding"],
    [["Fth1"], "geneSelection"],
)

vc3.layout((spatial3 | (lc3 / obs_sets3 / genes3)) / heatmap3)
vc3.widget()

**What to try in the widget above:**

- 🔎 Search for a gene in the Feature List (e.g., `EPCAM`, `CD44`, `ACTB`) — notice both the spatial spots and the heatmap update simultaneously
- 🖱️ Hover over a spot in the tissue image — the Status bar (if added) shows the spot ID and expression value
- 📋 Click a cell group in the OBS_SETS panel — highlights the corresponding spots in the spatial view

This is coordinated multiple views in action: a single user interaction propagates through the shared coordination space to all subscribed views.

## Exercises

### Exercise 1 — Pre-select a different gene

👉 **Modify the `vc3` cell above** to:

- **Change** the pre-selected gene from `"Fth1"` to a gene of your choice — try `"EPCAM"`, `"ACTB"`, or `"COL1A1"`

Re-run the cell to confirm the spots open with your chosen gene's expression visible.

---

### Exercise 2 — Add a status bar

👉 **Add a `STATUS` view** to the `vc3` dashboard so you can see hover information.

1. Add the view: `status = vc3.add_view(vt.STATUS, dataset=dataset3)`
2. Include `status` in the layout at the bottom — e.g., wrap the existing layout in a vertical split: `existing_layout / status`

> **Hint:** The status view should be very short. Make it the bottom row spanning the full width.

```python
# Template
status = vc3.add_view(vt.STATUS, dataset=dataset3)
vc3.layout(((spatial3 | (lc3 / obs_sets3 / genes3)) / heatmap3) / status)
```

---

### Exercise 3 — Side-by-side gene comparison for spatial data

👉 **Create two spatial views** showing the same tissue, each colored by a different gene.

Starting from the `vc3` pattern, create `spatial_a` and `spatial_b`, then use `add_coordination()` to give each its own `featureSelection` scope initialized to a different gene.

```python
vc4 = VitessceConfig(schema_version="1.0.18", name="Side-by-side genes")
# ... add dataset and wrapper (same as vc3) ...

spatial_a = vc4.add_view("spatialBeta", dataset=dataset4)
spatial_b = vc4.add_view("spatialBeta", dataset=dataset4)

# Give each spatial view its own independent gene selection
[scope_a] = vc4.add_coordination("featureSelection")
scope_a.set_value(["Fth1"])
[scope_b] = vc4.add_coordination("featureSelection")
scope_b.set_value(["EPCAM"])

spatial_a.use_coordination(scope_a)
spatial_b.use_coordination(scope_b)

vc4.link_views([spatial_a, spatial_b], ["obsColorEncoding", "obsType"], ["geneSelection", "spot"])

vc4.layout(spatial_a | spatial_b)
vc4.widget()
```

In [ ]:
# Exercise 3 — your answer here
vc4 = VitessceConfig(schema_version="1.0.18", name="Side-by-side gene comparison")

wrapper4 = SpatialDataWrapper(
    sdata_path=sdata_filepath,
    image_path="images/ST8059050_hires_image",
    obs_spots_path="shapes/ST8059050",
    table_path="tables/table",
    obs_feature_matrix_path="tables/table/X",
    region="ST8059050",
    coordinate_system="global",
    coordination_values={"obsType": "spot"},
)
dataset4 = vc4.add_dataset(name="Visium").add_object(wrapper4)

spatial_a = vc4.add_view("spatialBeta", dataset=dataset4)
spatial_b = vc4.add_view("spatialBeta", dataset=dataset4)

# TODO: add separate featureSelection scopes for each view
# [scope_a] = vc4.add_coordination("featureSelection")
# scope_a.set_value(["???"])
# [scope_b] = vc4.add_coordination("featureSelection")
# scope_b.set_value(["???"])
# spatial_a.use_coordination(scope_a)
# spatial_b.use_coordination(scope_b)

vc4.link_views([spatial_a, spatial_b], ["obsColorEncoding", "obsType"], ["geneSelection", wrapper4.obs_type_label])

vc4.layout(spatial_a | spatial_b)
vc4.widget()